# Ensemble

Объединяю лучшие CatBoost и XGBoost

CatBoost: 0.9198

XGBoost: 0.9202

Сначала проверяю простое усреднение вероятностей 50/50

In [1]:
import time
import pandas as pd

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

SEED = 20260916

train = pd.read_csv("../train.csv")
validation = pd.read_csv("../validation.csv")

feature_columns = [
    column
    for column in train.columns
    if column not in ["row_id", "target"]
]

X_train = train[feature_columns]
y_train = train["target"]

X_validation = validation[feature_columns]
y_validation = validation["target"]

print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Features:", len(feature_columns))

Train: (2520, 512)
Validation: (560, 512)
Features: 512


In [2]:
catboost_model = CatBoostClassifier(
    iterations=2000,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3,
    random_strength=2,
    random_seed=SEED,
    verbose=False,
)

xgboost_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.4,
    subsample=1.0,
    colsample_bytree=1.0,
    min_child_weight=1,
    random_state=SEED,
    n_jobs=-1,
    eval_metric="auc",
)

In [3]:
start = time.perf_counter()

catboost_model.fit(X_train, y_train)
xgboost_model.fit(X_train, y_train)

catboost_probability = catboost_model.predict_proba(
    X_validation
)[:, 1]

xgboost_probability = xgboost_model.predict_proba(
    X_validation
)[:, 1]

elapsed = time.perf_counter() - start

In [4]:
ensemble_probability = (
    0.5 * catboost_probability
    + 0.5 * xgboost_probability
)

catboost_auc = roc_auc_score(
    y_validation,
    catboost_probability,
)

xgboost_auc = roc_auc_score(
    y_validation,
    xgboost_probability,
)

ensemble_auc = roc_auc_score(
    y_validation,
    ensemble_probability,
)

results = pd.DataFrame([
    {
        "model": "CatBoost",
        "roc_auc": catboost_auc,
    },
    {
        "model": "XGBoost",
        "roc_auc": xgboost_auc,
    },
    {
        "model": "Ensemble 50/50",
        "roc_auc": ensemble_auc,
    },
])

results

,model,roc_auc
0,CatBoost,0.919821
1,XGBoost,0.920153
2,Ensemble 50/50,0.924732


### Вывод по Ensemble 50/50

Ансамбль CatBoost + XGBoost дал 0.9247

Это лучше обеих моделей по отдельности

Усреднение вероятностей: + качество

Дальше проверю разные веса CatBoost и XGBoost

In [5]:
ensemble_weight_results = []

for catboost_weight in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    xgboost_weight = 1 - catboost_weight

    ensemble_probability = (
        catboost_weight * catboost_probability
        + xgboost_weight * xgboost_probability
    )

    auc = roc_auc_score(
        y_validation,
        ensemble_probability,
    )

    ensemble_weight_results.append({
        "model": f"Ensemble {catboost_weight:.1f}/{xgboost_weight:.1f}",
        "roc_auc": auc,
        "catboost_weight": catboost_weight,
        "xgboost_weight": xgboost_weight,
    })

ensemble_weights_df = pd.DataFrame(
    ensemble_weight_results
)

ensemble_weights_df

,model,roc_auc,catboost_weight,xgboost_weight
0,Ensemble 0.2/0.8,0.922921,0.2,0.8
1,Ensemble 0.3/0.7,0.923967,0.3,0.7
2,Ensemble 0.4/0.6,0.924503,0.4,0.6
3,Ensemble 0.5/0.5,0.924732,0.5,0.5
4,Ensemble 0.6/0.4,0.924298,0.6,0.4
5,Ensemble 0.7/0.3,0.923380,0.7,0.3
6,Ensemble 0.8/0.2,0.922474,0.8,0.2


### Вывод по весам ансамбля

Лучший при весах - `0.5/0.5`: 0.9247

При смещении веса в сторону любой модели качество падало

Равное усреднение оказалось лучшим

Дальше `0.5/0.5`

### Rank averaging

Проверяю усреднение рангов предсказаний CatBoost и XGBoost

In [6]:
catboost_rank = pd.Series(
    catboost_probability
).rank(pct=True).to_numpy()

xgboost_rank = pd.Series(
    xgboost_probability
).rank(pct=True).to_numpy()

rank_ensemble_probability = (
    0.5 * catboost_rank
    + 0.5 * xgboost_rank
)

rank_ensemble_auc = roc_auc_score(
    y_validation,
    rank_ensemble_probability,
)

rank_results = pd.DataFrame([
    {
        "model": "Probability averaging",
        "roc_auc": ensemble_auc,
    },
    {
        "model": "Rank averaging",
        "roc_auc": rank_ensemble_auc,
    },
])

rank_results

,model,roc_auc
0,Probability averaging,0.924732
1,Rank averaging,0.925108


### Вывод по rank averaging

Rank averaging дал лучший результат: 0.9251

Обычное усреднение вероятностей дало 0.9247

Усреднение рангов немного улучшило качество

Дальше использую rank averaging